Use this notebook to create the Fold splits of the datasets: Nanododies or Nanobodies + antibodies.
Please run the ./NanoDesigner/jupyter_notebooks/process_datasets.ipynb prior runnning this script

In [1]:
# use nanodesigner1 kernel
from collections import defaultdict
import json
import os
from argparse import Namespace
import shutil
import numpy as np

In [ ]:
# Define default arguments
args = Namespace(
    data=".NanoDesigner/built_datasets/CDRH3_interacting_Nanobody_imgt.json", # path to json file "./CDRH3_interacting_Nanobody_imgt.json" 
    out_dir="./NanoDesigner/built_datasets/Nanobody_clustered_CDRH3", # folder to save all generated test/train/valid json files
    valid_ratio=0.1, # Ratio of validation test set
    test_ratio=0.1, # Ratio of test set
    cluster_seq="cdr", # choices=['cdr', 'antigen'],
    cdr=["H3"], # choices =  ["H3","H2","H1", "L3","L2","L1"], only used if cdr specified in the --cluster_seq
    filter="1*1", # H, L, A = heavy chain, light chain and antigen chains; "*"" = may or not be present; "1" = must be present 
    k_fold=10, # 'K fold dataset. -1 for not do k-fold. Note that if this is enabled, the test/valid ratio will be automatically calculated.
    seed=2022,
    benchmark=None #Path to benchmark json file. If this is enabled, Complexes from the benchmark will be used as test set and complexes from data will be used as train/valid.
                    #Note that complexes sharing clusters with those in the benchmark will be dropped. K fold will also be turned off.
)

In [6]:
def load_file(fpath):
    with open(fpath, 'r') as fin:
        lines = fin.read().strip().split('\n')
    items = [json.loads(s) for s in lines]
    return items

def save_file(lines, fpath):
    with open(fpath, 'w') as fout:
        fout.writelines(lines)

def exec_mmseq(cmd):
    r = os.popen(cmd)
    text = r.read()
    r.close()
    return text

def filter_flag(items, code):
    res = []
    for item in items:
        satisfy = True
        for permit, key in zip(code, ['heavy_chain', 'light_chain', 'antigen_chains', 'resolution']):
            if permit == '*':
                continue
            if key == 'resolution':
                satisfy = float(item[key]) < 4.0 if permit == '0' else float(item[key]) >= 4.0
            else:
                satisfy = len(item[key]) == 0 if permit == '0' else len(item[key]) > 0
            if not satisfy:
                break
        res.append(satisfy)
    return res


# Clustering function
def cluster_items(args, items, is_benchmark=None):
    if args.cluster_seq is None:
        clu2idx = {item['pdb']: [i] for i, item in enumerate(items)}
        return clu2idx
    
    tmp_dir = './tmp'
    os.makedirs(tmp_dir, exist_ok=True)
    fasta = os.path.join(tmp_dir, 'seq.fasta')
    
    with open(fasta, 'w') as fout:
        for item in items:
            pdb = item['pdb']
            seq = ""
            if args.cluster_seq == 'cdr':
                for cdr in args.cdr:
                    seq += item[f'cdr{cdr.lower()}_seq']
            elif args.cluster_seq == 'antigen':
                for ab in item['antigen_seqs']:
                    seq += ab
            fout.write(f'>{pdb}\n{seq}\n')
    
    db = os.path.join(tmp_dir, 'DB')
    cmd = f'mmseqs createdb {fasta} {db}'
    exec_mmseq(cmd)
    
    db_clustered = os.path.join(tmp_dir, 'DB_clu')
    cmd = f'mmseqs cluster {db} {db_clustered} {tmp_dir} --min-seq-id 0.95'  # adjust similarity threshold
    res = exec_mmseq(cmd)
    
    tsv = os.path.join(tmp_dir, 'DB_clu.tsv')
    cmd = f'mmseqs createtsv {db} {db} {db_clustered} {tsv}'
    exec_mmseq(cmd)

    # Parse the clustering results
    pdb2clu, clu2idx = {}, defaultdict(list)
    with open(tsv, 'r') as fin:
        entries = fin.read().strip().split('\n')
    for entry in entries:
        cluster, pdb = entry.split('\t')
        pdb2clu[pdb] = cluster
    
    for i, item in enumerate(items):
        cluster = pdb2clu[item['pdb']]
        clu2idx[cluster].append(i)
    
    shutil.rmtree(tmp_dir)  # Clean up temporary files
    return clu2idx


def setup_output_directory(args):
    data_dir = args.out_dir if args.out_dir else os.path.split(args.data)[0]
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
    return data_dir

# Split data function
def split_data(args, clu2idx, items, is_benchmark, data_dir):
    fnames = ['train', 'valid', 'test']
    
    if args.benchmark is not None:
        # If benchmark is provided, separate clusters into test and non-test
        benchmark_clusters, other_clusters = [], []
        for c in clu2idx:
            in_test = any(is_benchmark[i] for i in clu2idx[c])
            (benchmark_clusters if in_test else other_clusters).append(c)
        
        # Shuffle other clusters and determine split for validation
        np.random.shuffle(other_clusters)
        valid_len = int(len(other_clusters) * args.valid_ratio)
        valid_clusters, train_clusters = other_clusters[-valid_len:], other_clusters[:-valid_len]
        
        # Write to files
        for fname, clusters in zip(fnames, [train_clusters, valid_clusters, benchmark_clusters]):
            fpath = os.path.join(data_dir, f"{fname}.json")
            with open(fpath, 'w') as fout:
                for cluster in clusters:
                    for idx in clu2idx[cluster]:
                        items[idx]['cluster'] = cluster
                        fout.write(json.dumps(items[idx]) + '\n')
            print(f"Saved {len(clusters)} clusters to {fpath}")
    
    else:
        # Handle k-fold cross-validation split if specified
        clusters = list(clu2idx.keys())
        np.random.shuffle(clusters)
        
        if args.k_fold == -1:
            # Standard split without k-fold
            valid_len = int(len(clusters) * args.valid_ratio)
            test_len = int(len(clusters) * args.test_ratio)
            splits = [clusters[:-valid_len - test_len], clusters[-valid_len - test_len:-test_len], clusters[-test_len:]]
            
            for fname, split in zip(fnames, splits):
                fpath = os.path.join(data_dir, f"{fname}.json")
                with open(fpath, 'w') as fout:
                    for cluster in split:
                        for idx in clu2idx[cluster]:
                            items[idx]['cluster'] = cluster
                            fout.write(json.dumps(items[idx]) + '\n')
                print(f"Saved {len(split)} clusters to {fpath}")
        
        else:
            # Implement k-fold splitting
            print(f"Performing {args.k_fold}-fold cross-validation split")
            fold_size = len(clusters) // args.k_fold
            leftover = len(clusters) % args.k_fold
            
            for k in range(args.k_fold):
                fold_dir = os.path.join(data_dir, f'fold_{k}')
                os.makedirs(fold_dir, exist_ok=True)
                
                # Determine test clusters for this fold
                start = k * fold_size + min(k, leftover)
                end = start + fold_size + (1 if k < leftover else 0)
                test_clusters = clusters[start:end]
                
                # Remaining clusters go to train and validation
                train_val_clusters = clusters[:start] + clusters[end:]
                valid_len = int(len(train_val_clusters) * args.valid_ratio)
                valid_clusters = train_val_clusters[-valid_len:]
                train_clusters = train_val_clusters[:-valid_len]
                
                for fname, fold_clusters in zip(fnames, [train_clusters, valid_clusters, test_clusters]):
                    fpath = os.path.join(fold_dir, f"{fname}.json")
                    with open(fpath, 'w') as fout:
                        for cluster in fold_clusters:
                            for idx in clu2idx[cluster]:
                                items[idx]['cluster'] = cluster
                                fout.write(json.dumps(items[idx]) + '\n')
                    print(f"Fold {k}: Saved {len(fold_clusters)} clusters to {fpath}")



In [7]:
def main(args):
    np.random.seed(args.seed)
    items = load_file(args.data)
    
    flags = filter_flag(items, args.filter)
    items = [items[i] for i, flag in enumerate(flags) if flag]
    print(f'Valid entries after filtering: {len(items)}')
    
    if args.benchmark:
        benchmark = load_file(args.benchmark)
        flags = filter_flag(benchmark, args.filter)
        benchmark = [benchmark[i] for i, flag in enumerate(flags) if flag]
        is_benchmark = [False] * len(items) + [True] * len(benchmark)
        items.extend(benchmark)
        print(f'Benchmark entries: {len(benchmark)}')
    else:
        is_benchmark = None
    
    clu2idx = cluster_items(args, items, is_benchmark)
    data_dir = setup_output_directory(args)
    split_data(args, clu2idx, items, is_benchmark, data_dir)

main(args)

Valid entries after filtering: 391
Performing 10-fold cross-validation split
Fold 0: Saved 85 clusters to /home/rioszemm/NanoDesigner/built_datasets/Nanobody_clustered_CDRH3/fold_0/train.json
Fold 0: Saved 9 clusters to /home/rioszemm/NanoDesigner/built_datasets/Nanobody_clustered_CDRH3/fold_0/valid.json
Fold 0: Saved 11 clusters to /home/rioszemm/NanoDesigner/built_datasets/Nanobody_clustered_CDRH3/fold_0/test.json
Fold 1: Saved 85 clusters to /home/rioszemm/NanoDesigner/built_datasets/Nanobody_clustered_CDRH3/fold_1/train.json
Fold 1: Saved 9 clusters to /home/rioszemm/NanoDesigner/built_datasets/Nanobody_clustered_CDRH3/fold_1/valid.json
Fold 1: Saved 11 clusters to /home/rioszemm/NanoDesigner/built_datasets/Nanobody_clustered_CDRH3/fold_1/test.json
Fold 2: Saved 85 clusters to /home/rioszemm/NanoDesigner/built_datasets/Nanobody_clustered_CDRH3/fold_2/train.json
Fold 2: Saved 9 clusters to /home/rioszemm/NanoDesigner/built_datasets/Nanobody_clustered_CDRH3/fold_2/valid.json
Fold 2: 